In [10]:
import numpy as np
from time import time
from sklearn.metrics.pairwise import pairwise_kernels
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import sys
sys.path.append("../..")

In [11]:
from src.models.kernel_empirical_risk import KernelEmpiricalRisk, rkhs_norm
from src.data.load_data import load_data

Load data

In [12]:
input_points, output_points = load_data("friedman1")

In [13]:
train_input_points, test_input_points, train_output_points, test_output_points = (
    train_test_split(input_points, output_points, random_state=0)
)
calib_input_points, test_input_points, calib_output_points, test_output_points = (
    train_test_split(test_input_points, test_output_points, random_state=0)
)

scaler = StandardScaler()
scaler.fit(train_input_points)
train_input_points = scaler.transform(train_input_points)
calib_input_points = scaler.transform(calib_input_points)
test_input_points = scaler.transform(test_input_points)

Pick some random model weights

In [14]:
model_weights = np.random.rand(train_output_points.shape[0], train_output_points.shape[1])
model_weights.shape

# model_weights = np.zeros((train_output_points.shape[0], train_output_points.shape[1]))
# model_weights.shape

(375, 1)

Compute train Gram matrix on train and compute rkhs norm

In [15]:
gram_matrix = pairwise_kernels(train_input_points)

tic = time()
tmp = rkhs_norm(model_weights, gram_matrix)
tac = time()
print(tac - tic)

print(tmp)

0.0012369155883789062
2996.5711020903213


Compute empirical risk

In [16]:
lam = 0.1
loss_name="pseudo_huber"
loss_params={"alpha":1.}
empirical_risk = KernelEmpiricalRisk(loss_name, loss_params)
tic = time()
tmp = empirical_risk.empirical_risk(model_weights, gram_matrix, output_points, lam)
tac = time()
print(tac - tic)
print(tmp)

0.007962465286254883
350.1526118940939


In [17]:
gram_matrix[:, 1].shape

(375,)

Compute gradient of the empirical risk

In [19]:
tic = time()
empirical_risk_value, gradient_value = empirical_risk.empirical_risk_gradient(
    model_weights, gram_matrix,
    output_points, lam)
tac = time()
print(tac - tic)
print(empirical_risk_value)
print(gradient_value.shape)

0.007033824920654297
350.1526118940939
(375,)
